In [ ]:
import os
import glob
import cv2
import numpy as np
import zipfile
import gdown
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from transformers import pipeline

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [ ]:
# Change the URL if you use your own dataset link
url = 'https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL'
zip_out = "lipread_dataset.zip"
if not os.path.exists("dataset"):
    gdown.download(url, zip_out, quiet=False)
    with zipfile.ZipFile(zip_out, 'r') as z:
        z.extractall("dataset")
print("Dataset root folders (first 5):", os.listdir("dataset")[:5])


Downloading...
From (original): https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL
From (redirected): https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL&confirm=t&uuid=40dd33e6-8a17-4970-ac94-a28bc9aa7b49
To: /content/lipread_dataset.zip
100%|██████████| 423M/423M [00:11<00:00, 38.1MB/s]


Dataset root folders (first 5): ['data']


In [ ]:
IMG_SIZE = (50, 100)   # (height, width)
MAX_FRAMES = 75        # fixed number of frames per video

def load_video_fixed(path, max_frames=MAX_FRAMES, img_size=IMG_SIZE):
    h, w = img_size
    frames = np.zeros((max_frames, h, w), dtype=np.float32)  # preallocated
    cap = cv2.VideoCapture(path)
    idx = 0
    while idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame is None:
            break
        # convert to gray and resize (ensure consistent shape)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (w, h))
        frames[idx] = gray.astype(np.float32) / 255.0
        idx += 1
    cap.release()
    # if fewer frames read, remainder stays zeros (padding)
    return frames  # shape (MAX_FRAMES, h, w)


In [ ]:
def find_alignment_word(video_filename, align_folder):
    base = os.path.splitext(video_filename)[0]
    cand = os.path.join(align_folder, base + ".align")
    if os.path.exists(cand):
        with open(cand, "r", errors="ignore") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
            # find the first non-silence word token (safe heuristic)
            words = []
            for L in lines:
                parts = L.split()
                if len(parts) >= 3:
                    token = parts[-1]
                    if token.lower() not in ("sil", "sp", "pause"):
                        words.append(token)
            if words:
                return words[-1]  # last spoken token in that alignment
    return None

def load_dataset_grid(root_base="dataset"):
    video_dir = os.path.join(root_base, "data", "s1")
    align_dir = os.path.join(root_base, "data", "alignments", "s1")
    X, y_text, y_emotion, names = [], [], [], []
    if not os.path.isdir(video_dir):
        raise FileNotFoundError(f"Video folder not found: {video_dir}")
    for fname in sorted(os.listdir(video_dir)):
        if not fname.lower().endswith((".mpg", ".mp4", ".avi")):
            continue
        path = os.path.join(video_dir, fname)
        # load fixed-size padded frames
        frames = load_video_fixed(path)
        # get word from align file if possible
        word = find_alignment_word(fname, align_dir)
        if word is None:
            # fallback: use first 6 chars of filename as pseudo-label
            word = os.path.splitext(fname)[0][:6]
        X.append(frames)
        y_text.append(word.lower())
        y_emotion.append("neutral")  # dataset doesn't contain emotion labels -> default
        names.append(fname)
    X = np.array(X, dtype=np.float32)  # shape (N, MAX_FRAMES, h, w)
    y_text = np.array(y_text)
    y_emotion = np.array(y_emotion)
    print(f"Loaded {len(X)} videos from {video_dir}")
    return X, y_text, y_emotion, names

X, y_text, y_emotion, names = load_dataset_grid("dataset")
print("Example file:", names[0], "label:", y_text[0])
print("X shape:", X.shape)


Loaded 1000 videos from dataset/data/s1
Example file: bbaf2n.mpg label: now
X shape: (1000, 75, 50, 100)


In [ ]:
text_encoder = LabelEncoder()
y_text_enc = text_encoder.fit_transform(y_text)   # numeric labels for words

emotion_encoder = LabelEncoder()
y_emotion_enc = emotion_encoder.fit_transform(y_emotion)  # likely single class 'neutral'

X_train, X_test, y_text_train, y_text_test, y_emotion_train, y_emotion_test = train_test_split(
    X, y_text_enc, y_emotion_enc, test_size=0.15, random_state=42, stratify=y_text_enc
)

# expand channels dimension
X_train_np = np.expand_dims(X_train, -1).astype("float32")  # (N, frames, h, w, 1)
X_test_np = np.expand_dims(X_test, -1).astype("float32")

vocab_size = len(text_encoder.classes_)
num_emotions = max(1, len(emotion_encoder.classes_))

print("Train:", X_train_np.shape, "Test:", X_test_np.shape)
print("Vocab size (unique words):", vocab_size, "Emotion classes:", num_emotions)


Train: (850, 75, 50, 100, 1) Test: (150, 75, 50, 100, 1)
Vocab size (unique words): 4 Emotion classes: 1


In [ ]:
from tensorflow.keras import Model

def build_stable_model(vocab_size, num_emotions, input_shape=(MAX_FRAMES, IMG_SIZE[0], IMG_SIZE[1], 1)):
    inp = layers.Input(shape=input_shape, name="video_input")
    x = layers.Conv3D(32, (3,3,3), activation='relu', padding='same')(inp)
    x = layers.MaxPool3D((1,2,2))(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv3D(64, (3,3,3), activation='relu', padding='same')(x)
    x = layers.MaxPool3D((1,2,2))(x)
    x = layers.BatchNormalization()(x)

    # collapse spatial dims per frame to make sequence features
    x = layers.TimeDistributed(layers.Flatten())(x)  # (batch, frames, features)
    x = layers.Bidirectional(layers.GRU(128, return_sequences=False))(x)  # final sequence vector

    # Word classification head
    word_out = layers.Dense(vocab_size, activation="softmax", name="word_output")(x)

    # Emotion head (softmax if multiple, else sigmoid)
    if num_emotions > 1:
        emotion_out = layers.Dense(num_emotions, activation="softmax", name="emotion_output")(x)
        loss_emotion = "sparse_categorical_crossentropy"
    else:
        # single class -> make it binary format (still use sigmoid)
        emotion_out = layers.Dense(1, activation="sigmoid", name="emotion_output")(x)
        loss_emotion = "binary_crossentropy"

    model = Model(inputs=inp, outputs=[word_out, emotion_out])
    model.compile(
        optimizer="adam",
        loss={"word_output": "sparse_categorical_crossentropy", "emotion_output": loss_emotion},
        metrics={"word_output": "accuracy", "emotion_output": "accuracy"}
    )
    return model

model = build_stable_model(vocab_size, num_emotions, input_shape=X_train_np.shape[1:])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 75, 50,    │          0 │ -                 │
│ (InputLayer)        │ 100, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d (Conv3D)     │ (None, 75, 50,    │        896 │ video_input[0][0] │
│                     │ 100, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d       │ (None, 75, 25,    │          0 │ conv3d[0][0]      │
│ (MaxPooling3D)      │ 50, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 75, 25,    │        128 │ max_pooling3d[0]… │
│ (BatchNormalizatio… │ 50, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_1 (Conv3D)   │ (None, 75, 25,    │     55,360 │ batch_normalizat… │
│                     │ 50, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_1     │ (None, 75, 12,    │          0 │ conv3d_1[0][0]    │
│ (MaxPooling3D)      │ 25, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 75, 12,    │        256 │ max_pooling3d_1[… │
│ (BatchNormalizatio… │ 25, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 75, 19200) │          0 │ batch_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 256)       │ 14,845,440 │ time_distributed… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_output (Dense) │ (None, 4)         │      1,028 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emotion_output      │ (None, 1)         │        257 │ bidirectional[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,903,365 (56.85 MB)

 Trainable params: 14,903,173 (56.85 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# Train on small subset to avoid long runs; increase if you have GPU and time
train_count = min(200, X_train_np.shape[0])
X_tr = X_train_np[:train_count]
y_word_tr = y_text_train[:train_count]
y_emo_tr = y_emotion_train[:train_count]

history = model.fit(
    X_tr,
    {"word_output": y_word_tr, "emotion_output": y_emo_tr},
    validation_split=0.1,
    epochs=3,
    batch_size=8,
    verbose=2
)


Epoch 1/3
23/23 - 793s - 34s/step - emotion_output_accuracy: 0.9611 - emotion_output_loss: 0.0508 - loss: 1.9246 - val_emotion_output_accuracy: 1.0000 - val_emotion_output_loss: 4.9966e-04 - val_loss: 1.5825 - val_word_output_accuracy: 0.4000 - val_word_output_loss: 1.5216 - word_output_accuracy: 0.2389 - word_output_loss: 1.8604
Epoch 2/3
23/23 - 753s - 33s/step - emotion_output_accuracy: 1.0000 - emotion_output_loss: 1.2185e-05 - loss: 1.4204 - val_emotion_output_accuracy: 1.0000 - val_emotion_output_loss: 3.1548e-06 - val_loss: 1.5287 - val_word_output_accuracy: 0.2500 - val_word_output_loss: 1.5080 - word_output_accuracy: 0.2000 - word_output_loss: 1.4208
Epoch 3/3
23/23 - 808s - 35s/step - emotion_output_accuracy: 1.0000 - emotion_output_loss: 1.4612e-05 - loss: 1.4342 - val_emotion_output_accuracy: 1.0000 - val_emotion_output_loss: 1.1897e-06 - val_loss: 1.6377 - val_word_output_accuracy: 0.4000 - val_word_output_loss: 1.6208 - word_output_accuracy: 0.2833 - word_output_loss: 1.4

In [ ]:
import heapq
def beam_search_decoder(probs_seq, beam_width=3):
    # probs_seq: numpy array shape (T, V) — probabilities per time-step
    sequences = [([], 0.0)]
    for t in range(probs_seq.shape[0]):
        all_candidates = []
        for seq, score in sequences:
            for c in range(probs_seq.shape[1]):
                p = probs_seq[t, c]
                new_seq = seq + [c]
                new_score = score - np.log(p + 1e-12)
                all_candidates.append((new_seq, new_score))
        sequences = sorted(all_candidates, key=lambda tup: tup[1])[:beam_width]
    return sequences[0][0]



In [ ]:
# pick a test sample
idx = 0
x_sample = X_test_np[idx:idx+1]  # shape (1, frames, h, w, 1)
true_word = text_encoder.inverse_transform([y_text_test[idx]])[0]
true_emotion = emotion_encoder.inverse_transform([y_emotion_test[idx]])[0] if num_emotions > 1 else emotion_encoder.inverse_transform([0])[0]

pred_word_probs, pred_emo = model.predict(x_sample, verbose=0)

# word prediction (direct classifier)
pred_word_idx = int(np.argmax(pred_word_probs[0]))
predicted_word = text_encoder.inverse_transform([pred_word_idx])[0]

# emotion: map prediction to label
if num_emotions > 1:
    pred_emo_idx = int(np.argmax(pred_emo[0]))
    predicted_emotion = emotion_encoder.inverse_transform([pred_emo_idx])[0]
else:
    # single class fallback
    predicted_emotion = emotion_encoder.inverse_transform([0])[0]

# Demonstrate beam search by creating a repeated prob sequence (demo purpose)
probs_seq = np.tile(pred_word_probs[0], (MAX_FRAMES, 1))  # shape (T, V)
decoded_seq = beam_search_decoder(probs_seq, beam_width=5)
# decoded_seq is a list of token indices per time-step; collapse to most frequent token:
most_common_idx = max(set(decoded_seq), key=decoded_seq.count)
beam_decoded_word = text_encoder.inverse_transform([most_common_idx])[0]

# text-based emotion classifier (transformer)
try:
    text_emotion_pipeline = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=1)
    text_emotion_label = text_emotion_pipeline(predicted_word)[0]['label']
except Exception as e:
    text_emotion_label = "N/A"

print("====== Final Prediction Result ======")
print("Predicted Word (classifier) :", predicted_word)
print("Beam-decoded Word (demo)    :", beam_decoded_word)
print("True Word                   :", true_word)
print("Predicted Emotion (model)   :", predicted_emotion)
print("Text-based Emotion (pipe)   :", text_emotion_label)
print("True Emotion                :", true_emotion)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


====== Final Prediction Result ======
Predicted Word (classifier) : now
Beam-decoded Word (demo)    : now
True Word                   : now
Predicted Emotion (model)   : neutral
Text-based Emotion (pipe)   : N/A
True Emotion                : neutral


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving test.mp4 to test.mp4


In [ ]:
import cv2
import numpy as np
from google.colab import files

In [ ]:
uploaded = files.upload()   # Upload a .mp4, .avi, or .mpg file
video_path = list(uploaded.keys())[0]
print(f"✅ Uploaded: {video_path}")

Saving test.mp4 to test (1).mp4
✅ Uploaded: test (1).mp4


In [ ]:
def load_video(path):
    cap = cv2.VideoCapture(path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Convert to grayscale
        try:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        except:
            frame = np.mean(frame, axis=-1).astype(np.uint8)

        # Resize to (height=50, width=100)
        frame = cv2.resize(frame, (100, 50))
        frames.append(frame)

        # Stop once MAX_FRAMES reached
        if len(frames) >= MAX_FRAMES:
            break

    cap.release()

    # Pad short videos with black frames
    while len(frames) < MAX_FRAMES:
        frames.append(np.zeros((50, 100), dtype=np.uint8))

    # Convert list to numpy array and normalize
    frames = np.stack(frames, axis=0).astype("float32") / 255.0
    frames = np.expand_dims(frames, axis=-1)  # shape: (75, 50, 100, 1)
    return frames


In [ ]:
test_video = load_video(video_path)
test_video = np.expand_dims(test_video, axis=0)
print("🎞 Video preprocessed successfully. Shape:", test_video.shape)


🎞 Video preprocessed successfully. Shape: (1, 75, 50, 100, 1)


In [ ]:
def beam_search_decoder(predictions, beam_width=5):
    import heapq
    predictions = np.atleast_2d(predictions)  # ensures shape (T, V)
    T, V = predictions.shape

    sequences = [([], 0.0)]  # (sequence, log-probability)
    for t in range(T):
        all_candidates = []
        row = predictions[t]
        for seq, score in sequences:
            for j, prob in enumerate(row):
                new_seq = seq + [j]
                new_score = score - np.log(prob + 1e-9)
                all_candidates.append((new_seq, new_score))
        sequences = heapq.nsmallest(beam_width, all_candidates, key=lambda x: x[1])
    return sequences[0][0]


In [ ]:
preds_text, preds_emotion = model.predict(test_video)

# Beam search decoding for text
beam_indices = beam_search_decoder(preds_text[0])
decoded_word = ''.join([text_encoder.inverse_transform([i])[0] for i in beam_indices if i < len(text_encoder.classes_)])

# Emotion prediction
emotion_label = emotion_encoder.inverse_transform([np.argmax(preds_emotion[0])])

print("🗣 Predicted Word:", decoded_word)
print("😊 Predicted Emotion:", emotion_label[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
🗣 Predicted Word: please
😊 Predicted Emotion: neutral
